# Cybercities: data download

This notebook downloads every raw data file the project needs and packs them into one zip, `cybercities_raw.zip`.

**How to use:** click **Runtime → Run all**. It takes about 3-5 minutes. Each step prints `OK` or `FAILED`; a failure in one step does not stop the others. At the end your browser downloads the zip. Upload that zip back to Claude.

Note: the Census API key below is personal. Remove it before sharing this notebook anywhere public.

In [ ]:
import io, os, zipfile, requests, pandas as pd

CENSUS_KEY = "YOUR_CENSUS_API_KEY"  # personal key: remove before publishing

# The 40 largest US metro areas (CBSA codes, 2023 OMB definitions)
CBSA = ["35620","31080","16980","19100","26420","12060","47900","33100","37980","38060",
        "14460","40140","41860","19820","42660","33460","45300","41740","19740","36740",
        "16740","12580","41180","41700","12420","38900","40900","38300","29820","17140",
        "28140","18140","26900","34980","17460","41940","47260","27260","39300","39580"]

OUT = "cybercities_raw"
os.makedirs(OUT, exist_ok=True)

# Some government sites refuse requests that don't look like a browser
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0 Safari/537.36",
           "Accept": "*/*", "Accept-Language": "en-US,en;q=0.9"}
GEO = "metropolitan%20statistical%20area/micropolitan%20statistical%20area"
status = {}

def census_get(url):
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    rows = r.json()
    return pd.DataFrame(rows[1:], columns=rows[0])

def download(url):
    r = requests.get(url, headers=HEADERS, timeout=300)
    r.raise_for_status()
    return r.content

print("Setup OK")

## 1. Census ACS 2024: commuting, Gini index, rent burden

In [ ]:
try:
    variables = ",".join(["NAME", "B01003_001E", "B19083_001E",
        "B08301_001E", "B08301_010E", "B08301_019E", "B08301_021E", "B08013_001E",
        "B25070_001E", "B25070_007E", "B25070_008E", "B25070_009E", "B25070_010E", "B25070_011E"])
    url = f"https://api.census.gov/data/2024/acs/acs1?get={variables}&for={GEO}:*&key={CENSUS_KEY}"
    acs = census_get(url)
    acs.to_csv(f"{OUT}/acs_2024.csv", index=False)
    status["ACS"] = f"OK ({len(acs)} areas)"
except Exception as e:
    status["ACS"] = f"FAILED: {e}"
print(status["ACS"])

## 2. Census Gazetteer 2024: land area of each metro

In [ ]:
try:
    raw = download("https://www2.census.gov/geo/docs/maps-data/data/gazetteer/2024_Gazetteer/2024_Gaz_cbsa_national.zip")
    z = zipfile.ZipFile(io.BytesIO(raw))
    name = [n for n in z.namelist() if n.endswith(".txt")][0]
    gaz = pd.read_csv(z.open(name), sep="\t", dtype=str, encoding="latin-1")
    gaz.columns = gaz.columns.str.strip()
    gaz.to_csv(f"{OUT}/gazetteer_cbsa_2024.csv", index=False)
    status["Gazetteer"] = f"OK ({len(gaz)} areas)"
except Exception as e:
    status["Gazetteer"] = f"FAILED: {e}"
print(status["Gazetteer"])

## 3. County Business Patterns 2023: businesses by industry in each metro

In [ ]:
try:
    parts, failed = [], []
    for c in CBSA:
        url = (f"https://api.census.gov/data/2023/cbp?get=NAME,NAICS2017,NAICS2017_LABEL,ESTAB,EMP"
               f"&for={GEO}:{c}&key={CENSUS_KEY}")
        try:
            parts.append(census_get(url))
        except Exception as e:
            failed.append(c)
    cbp = pd.concat(parts, ignore_index=True)
    cbp.to_csv(f"{OUT}/cbp_2023.csv", index=False)
    status["CBP"] = f"OK ({len(cbp)} rows, {len(parts)} metros)" + (f", missing: {failed}" if failed else "")
except Exception as e:
    status["CBP"] = f"FAILED: {e}"
print(status["CBP"])

## 4. Business Formation Statistics: new business applications by county (2005-2025)

In [ ]:
try:
    raw = download("https://www.census.gov/econ/bfs/xlsx/bfs_county_apps_annual.xlsx")
    open(f"{OUT}/bfs_county_apps_annual.xlsx", "wb").write(raw)
    status["BFS"] = f"OK ({len(raw)//1024} KB)"
except Exception as e:
    status["BFS"] = f"FAILED: {e}"
print(status["BFS"])

## 5. OMB 2023 delineation file: which counties belong to which metro

In [ ]:
try:
    raw = download("https://www2.census.gov/programs-surveys/metro-micro/geographies/reference-files/2023/delineation-files/list1_2023.xlsx")
    open(f"{OUT}/list1_2023.xlsx", "wb").write(raw)
    status["Delineation"] = f"OK ({len(raw)//1024} KB)"
except Exception as e:
    status["Delineation"] = f"FAILED: {e}"
print(status["Delineation"])

## 6. BEA: GDP by metro area (with a FRED fallback)

In [ ]:
try:
    raw = download("https://apps.bea.gov/regional/zip/CAGDP1.zip")
    z = zipfile.ZipFile(io.BytesIO(raw))
    name = [n for n in z.namelist() if "ALL_AREAS" in n and n.endswith(".csv")][0]
    bea = pd.read_csv(z.open(name), dtype=str, encoding="latin-1")
    bea["GeoFIPS"] = bea["GeoFIPS"].str.replace('"', "").str.strip()
    bea = bea[bea["GeoFIPS"].isin(CBSA)]
    bea.to_csv(f"{OUT}/bea_gdp.csv", index=False)
    status["BEA"] = f"OK ({name}, {bea['GeoFIPS'].nunique()} metros)"
except Exception as e:
    status["BEA"] = f"FAILED: {e}"
    try:
        parts = []
        for c in CBSA:
            s = pd.read_csv(io.BytesIO(download(f"https://fred.stlouisfed.org/graph/fredgraph.csv?id=NGMP{c}")))
            s.columns = ["date", "gdp_millions"]
            s["cbsa_code"] = c
            parts.append(s)
        pd.concat(parts).to_csv(f"{OUT}/fred_gdp.csv", index=False)
        status["BEA"] += " | FRED fallback OK"
    except Exception as e2:
        status["BEA"] += f" | FRED fallback FAILED: {e2}"
print(status["BEA"])

## 7. BLS OEWS May 2025: computer and math jobs by metro

In [ ]:
try:
    raw = download("https://www.bls.gov/oes/special-requests/oesm25ma.zip")
    z = zipfile.ZipFile(io.BytesIO(raw))
    name = [n for n in z.namelist() if "MSA" in n.upper() and n.endswith(".xlsx")][0]
    oews = pd.read_excel(z.open(name), dtype=str)
    oews.columns = oews.columns.str.upper()
    oews = oews[oews["OCC_CODE"].isin(["00-0000", "15-0000"])]
    oews.to_csv(f"{OUT}/oews_2025.csv", index=False)
    status["OEWS"] = f"OK ({name}, {len(oews)} rows)"
except Exception as e:
    status["OEWS"] = (f"FAILED: {e}  ->  manual fix: open https://www.bls.gov/oes/special-requests/oesm25ma.zip "
                      "in your browser and upload that zip to Claude together with the other one")
print(status["OEWS"])

## 8. National Transit Database 2024: rail route miles

In [ ]:
for label, ds in [("NTD_by_mode", "wwdp-t4re"), ("NTD_summary", "9npq-v85z")]:
    try:
        raw = download(f"https://data.transportation.gov/api/views/{ds}/rows.csv?accessType=DOWNLOAD")
        open(f"{OUT}/{label.lower()}.csv", "wb").write(raw)
        status[label] = f"OK ({len(raw)//1024} KB)"
    except Exception as e:
        status[label] = f"FAILED: {e}"
    print(label, status[label])

## 9. Pack everything and download

In [ ]:
with zipfile.ZipFile("cybercities_raw.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(OUT):
        z.write(f"{OUT}/{f}", f)

print("SUMMARY")
for k, v in status.items():
    print(f"  {k}: {v}")
print(f"\nZip size: {os.path.getsize('cybercities_raw.zip')//1024} KB")

from google.colab import files
files.download("cybercities_raw.zip")